# 09 - Spatial displacement (Stage 11)

All estimation is done by `src/models/displacement_model.py`; this notebook only calls it and presents the results. Estimates are **provisional** while limitation M8 is open (see README).

**Sign convention.** Outcomes are crime counts, so every effect is a *change in crimes* (positive = more crime). The rings are disjoint, so
`tau_net = tau_direct + tau_displacement`. Displacement proportion = `-tau_displacement / tau_direct` (share of the direct effect offset by an opposite-signed ring effect; undefined when `tau_direct = 0`).

In [1]:
import sys, math
from pathlib import Path
import numpy as np, pandas as pd
ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.models import displacement_model as s11
from src.models import did_model
pd.set_option('display.width', 200)

## 1. Data overview

In [2]:
df = did_model.load_panel(s11.CAUSAL_FILE)
print(df.shape)
print(df.groupby(['role','period'])[['crime_100m','crime_250m']].mean().round(4))
print('pairs', df.pair_id.nunique(), 'units', df.unit_id.nunique(), 'clusters', df.treatment_h3_res7.nunique())

(120306, 31)
             crime_100m  crime_250m
role period                        
C    during      0.1622      0.9298
     post        0.2113      1.2540
     pre         0.2190      1.2628
T    during      0.1679      0.9872
     post        0.2336      1.3389
     pre         0.2256      1.3441
pairs 20051 units 40102 clusters 193


## 2-6. Direct, displacement, net effects, uncertainty, displacement proportion
One call to the Stage 11 module (the same call the CLI makes).

In [3]:
table, results, pairs = s11.run(s11.CAUSAL_FILE)
table[['effect_name','estimate','standard_error','variance','ci_lower','ci_upper','p_value']]

,effect_name,estimate,standard_error,variance,ci_lower,ci_upper,p_value
0,direct_during,-0.000848,0.007188,0.000052,-0.014937,0.013241,0.906112
1,displacement_during,-0.023889,0.018351,0.000337,-0.059857,0.012078,0.192992
2,net_during,-0.024737,0.020196,0.000408,-0.064319,0.014846,0.220624
3,displacement_proportion_during,-28.176471,238.284119,56779.321162,-495.204761,438.851820,0.905872
4,direct_post,0.015760,0.007588,0.000058,0.000887,0.030633,0.037816
5,displacement_post,0.003740,0.019409,0.000377,-0.034301,0.041782,0.847182
6,net_post,0.019500,0.021663,0.000469,-0.022959,0.061959,0.368034
7,displacement_proportion_post,-0.237342,1.223270,1.496388,-2.634906,2.160222,0.846158


### Net-effect variance and covariance

In [4]:
for period, r in results.items():
    va, vb, c = r['direct'][1], r['displacement'][1], r['covariance']
    print(period, f"Var(direct)={va:.3e} Var(disp)={vb:.3e} Cov={c:.3e} Var(net)={r['net'][1]:.3e}",
          f"| independence would give {va+vb:.3e}")

during Var(direct)=5.167e-05 Var(disp)=3.368e-04 Cov=9.711e-06 Var(net)=4.079e-04 | independence would give 3.884e-04
post Var(direct)=5.758e-05 Var(disp)=3.767e-04 Cov=1.749e-05 Var(net)=4.693e-04 | independence would give 4.343e-04


## 7. Sign convention

In [5]:
print(s11.SIGN_CONVENTION)
print(table[['effect_name','outcome_ring']].drop_duplicates('outcome_ring').to_string(index=False))

tau = change in crime counts (positive = more crime); net = direct + displacement; displacement_proportion = -displacement / direct
                   effect_name                      outcome_ring
                 direct_during              0-100 m (crime_100m)
           displacement_during       100-250 m ring (crime_250m)
                    net_during 0-250 m (crime_100m + crime_250m)
displacement_proportion_during     100-250 m relative to 0-100 m


## 8. Diagnostics
The module asserts that direct and displacement estimates and SEs equal Stage 9's `mean_with_cluster_se`, that the paired means equal the two-way FE coefficients, and that the net variance equals the cluster-robust variance of the summed paired differences. `run` raised no error above, so all passed.

In [6]:
saved = pd.read_csv(ROOT / 'outputs' / s11.OUTPUT_FILENAME)
print('saved CSV matches fresh run:', np.allclose(saved.select_dtypes('number').fillna(0), table.select_dtypes('number').fillna(0)))
print('finite (excl. proportion rows):', np.isfinite(table[~table.effect_name.str.startswith('displacement_proportion')][['estimate','standard_error']].to_numpy()).all())

saved CSV matches fresh run: True
finite (excl. proportion rows): True


## 9. Interpretation
Read the printed table: no ring effect or net effect has a 95% interval excluding zero in either period; only the direct post-period effect is marginally distinguishable from zero. The displacement proportion has a very wide interval (its denominator, the direct effect, is close to zero), so it is not informative here. Net effects are small relative to their standard errors. Findings remain provisional (M1-M8; contamination M3 is large and could bias the ring estimates).